In this notebook, we'll build a bigram language model from scratch, and use it to generate some random text. 

# Data: Alice in Wonderland

For this notebook, we will use Lewis Carroll's "Alice in Wonderland", available on Project Gutenberg here: http://www.gutenberg.org/ebooks/11
Please get the plain text version, and store it in the same directory as this notebook as a file ```11-0.txt```

## Preprocessing the data

We split up the text into sentences, then the sentences into words. As in the "Green eggs and ham" corpus example, we put sentence beginning and end markers around each sentence.

In [1]:
# We read in Alice in Wonderland as a single long string, which we 
# tokenize into sentences. 
# Warning: This may take a while on some computers
f = open("11-0.txt")
alice_text = f.read()
f.close()

import nltk

alice_sents = nltk.sent_tokenize(alice_text)
alice_sents_tokenized = [ nltk.word_tokenize(s) for s in alice_sents]

alice_corpus = []
for st in alice_sents_tokenized:
    alice_corpus = alice_corpus + ["<s>"] + st + ["</s>"] 

alice_bigrams = list(nltk.bigrams(alice_corpus))

We inspect the corpus:

In [2]:
for word in alice_corpus[400:450]: 
    print(word, end =" ")

_very_ remarkable in that ; nor did Alice think it so _very_ much out of the way to hear the Rabbit say to itself , “ Oh dear ! </s> <s> Oh dear ! </s> <s> I shall be late ! ” ( when she thought it over afterwards , 

And the bigrams:

In [3]:
alice_bigrams[600:610]

[(',', 'and'),
 ('and', 'then'),
 ('then', 'dipped'),
 ('dipped', 'suddenly'),
 ('suddenly', 'down'),
 ('down', ','),
 (',', 'so'),
 ('so', 'suddenly'),
 ('suddenly', 'that'),
 ('that', 'Alice')]

# Counting words, and computing relative frequencies

## Conditional Frequency Distribution objects in NLTK

The Natural Language Toolkit has data types and functions that make life easier for us when we want to count word n-grams and compute their probabilities using Maximum Likelihood Estimation.

The FreqDist object is a dictionary made for word counting (basically the same as a Counter). Here is an example: 

In [4]:
fd = nltk.FreqDist(alice_bigrams)
fd.most_common(20)

[(('</s>', '<s>'), 1101),
 (('.', '</s>'), 893),
 ((',', 'and'), 455),
 (('<s>', '“'), 446),
 ((',', '”'), 403),
 (('”', 'said'), 331),
 (('!', '”'), 282),
 (('”', '“'), 241),
 (('’', 't'), 216),
 (('said', 'the'), 206),
 (('’', 's'), 202),
 ((',', '“'), 200),
 (('“', 'I'), 163),
 (('of', 'the'), 155),
 (('?', '”'), 153),
 (('!', '</s>'), 153),
 (('I', '’'), 128),
 (('said', 'Alice'), 115),
 (('in', 'a'), 100),
 (('in', 'the'), 92)]

The ConditionalFreqDist object counts pairs of items, for example: Given that the first word in a pair is 'said', how often is the second word 'Alice'? How often is it 'the'?

In [5]:
# This makes the object
cfd = nltk.ConditionalFreqDist(alice_bigrams)
# This looks up the counts of bigrams starting in "said"
cfd["said"]

FreqDist({'the': 206, 'Alice': 115, 'to': 39, ',': 28, 'in': 7, 'this': 6, 'nothing': 6, '“': 5, '.': 4, 'very': 3, ...})

As you can see, the book has 206 occurrences of "said the", 115 of "said Alice", and so on. 

## Computing bigram frequencies and probabilities 
There is one more data structure we need, NLTK's ConditionalProbDist. This, finally, turns the conditional counts into probabilities. Here is how to build it for Alice in Wonderland: We give it the conditional frequencies, and say that we want MLE probabilities.


In [6]:
# this makes the object
cp = nltk.ConditionalProbDist(cfd, nltk.MLEProbDist)
# this is p(Alice|said)
cp["said"].prob("Alice")

0.25386313465783666

In [7]:
# Here are all the words that could follow "said"
print("Words that can come after 'said':\n", list(cp["said"].samples()))
# For each word in samples(), we can get its probability
# using prob()
print("And here are their probabilities:")
for word in cp["said"].samples():
    print(word, cp["said"].prob(word))

Words that can come after 'said':
 ['aloud', 'Alice', ',', 'anxiously', 'poor', 'this', 'these', 'to', 'nothing', 'in', '.', 'the', 'his', 'right', 'very', 'pig', 'with', '“', 'Five', 'Seven', 'severely', 'it', 'Two', 'a', '‘', 'after', ';', 'as', 'I', ':', 'so', 'one', 'that', 'gravely', 'her']
And here are their probabilities:
aloud 0.002207505518763797
Alice 0.25386313465783666
, 0.06181015452538632
anxiously 0.002207505518763797
poor 0.002207505518763797
this 0.013245033112582781
these 0.002207505518763797
to 0.08609271523178808
nothing 0.013245033112582781
in 0.01545253863134658
. 0.008830022075055188
the 0.45474613686534215
his 0.002207505518763797
right 0.002207505518763797
very 0.006622516556291391
pig 0.002207505518763797
with 0.004415011037527594
“ 0.011037527593818985
Five 0.006622516556291391
Seven 0.002207505518763797
severely 0.002207505518763797
it 0.002207505518763797
Two 0.002207505518763797
a 0.006622516556291391
‘ 0.002207505518763797
after 0.002207505518763797
; 0.0

# Generating text: Probabilistically sampling the next word at each step

Say you roll one fair die. Then each side has a probability of 1/6 of coming up. We can describe this with a *categorical distribution* -- or with a *multinomial distribution*, which describes what happens when you roll k fair dice at once, where we just set k to be 1. The Python *scipy* package has an object that describes a multinomial distribution, called ```stats.multinomial```. It has a function called ```rvs()``` that we can use to roll a die once: It will give us a probabilistic answer, a different one each time we run it. 

Run the following box a few times: 

In [8]:
# Let's roll one die
from scipy import stats

# we have 6 possible outcomes, each with a probability of 1/6
outcomes = [1,2,3,4,5,6]
outcome_probs = [1/6, 1/6, 1/6, 1/6, 1/6, 1/6]
# we only roll one die
num_dice = 1

result = stats.multinomial.rvs(num_dice, outcome_probs)
print("I just rolled one die. Here is what I got.")
# the result we get is a list of 0's and 1's. We get a 1 for
# the number that we rolled
print("Here is what the result looks like out of the box:", result)
# the index of the outcome we got is the 
# place in the result that has a 1
index_of_outcome = list(result).index(1)
print("And here is how we can interpret this:")
print("I rolled a", outcomes[index_of_outcome])


I just rolled one die. Here is what I got.
Here is what the result looks like out of the box: [0 0 0 1 0 0]
And here is how we can interpret this:
I rolled a 4


We can use the same method to probabilistically draw a word. Say we want to draw a bigram that starts in "said...". And say we use the probabilities that we got from the Alice in Wonderland corpus above. Then we can do this:

In [9]:
# possible outcomes: all the words that can follow after "said"
outcomes = list(cp["said"].samples())
# their probabilities, according to the ConditionalProbDist object
outcome_probs = [cp["said"].prob(word) for word in outcomes]
# we only generate a single next word
num_dice = 1

result = stats.multinomial.rvs(num_dice, outcome_probs)
print("I just drew a bigram starting in 'said...'. Here is what I got.")
index_of_outcome = list(result).index(1)
print("said", outcomes[index_of_outcome])

I just drew a bigram starting in 'said...'. Here is what I got.
said with


Now we are ready to generate a longer stretch of text. We start with the sentence start symbol

In [10]:
currentword = "<s>"
print(currentword, end = " ")
# The language model knows, for each word w, 
# the conditional probability P(w|currentword).
# We use that to generate text:
# we start with the sentence-start symbol
for i in range(50):
    # get all the possible words that can follow the current word
    outcomes = list(cp[currentword].samples())
    # and get their probabilities
    outcome_probs = [cp[currentword].prob(word) for word in outcomes]
    # we only sample one next word
    num_dice = 1
    # sample the next word
    result = stats.multinomial.rvs(num_dice, outcome_probs)
    # which outcome index did we sample?
    index_of_outcome = list(result).index(1)
    # The new current word is the word at that index
    currentword = outcomes[ index_of_outcome]
    # print it, and repeat
    print(currentword,end = " ")

<s> “ Yes , Alice , no one a-piece , two sides of the same thing , when suddenly a new eBooks , but was gone to the procession came skimming out what does very difficult question certainly : all advance twice— ” There was lying on which the tops of 